<a href="https://colab.research.google.com/github/quasarx-snips/devjams_lunap/blob/main/Module_1/ResNet_Hazard_Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip -q install -U segmentation-models-pytorch datasets huggingface_hub tqdm

import os, sys, random, time, math, glob, zipfile, requests
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from tqdm.auto import tqdm

SEED = 95 #dont ask why
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #if on gcolab set it from runtime then t4gpu
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print("Setting to cuDNN...")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: None (Running on CPU)")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", device)




Setting to cuDNN...
GPU: Tesla T4
Python: 3.13.15
PyTorch: 2.11.0+cu128
Device: cuda


In [4]:
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CKPT_DIR = "/content/drive/MyDrive/lunap_hazard_checkpoints"
except Exception as e:
    CKPT_DIR = "/content/checkpoints"
    print("Drive mount skipped:", e)

os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoint directory:", CKPT_DIR)


Mounted at /content/drive
Checkpoint directory: /content/drive/MyDrive/lunap_hazard_checkpoints


In [5]:
from datasets import load_dataset

print("Loading crater dataset...")
crater_ds = load_dataset("gremlin97/crater_binary_segmentation")
print(crater_ds)
for split in ("train", "val", "test"):
    if split in crater_ds:
        print(f"{split:>5}: {len(crater_ds[split]):,} samples")


Loading crater dataset...


README.md:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  356MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  355MB            

...

data/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/900 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/900 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'mask', 'width', 'height', 'class_labels'],
        num_rows: 3600
    })
    test: Dataset({
        features: ['image', 'mask', 'width', 'height', 'class_labels'],
        num_rows: 900
    })
    val: Dataset({
        features: ['image', 'mask', 'width', 'height', 'class_labels'],
        num_rows: 900
    })
})
train: 3,600 samples
  val: 900 samples
 test: 900 samples
